In [1]:
!pip install wilds --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.7 MB/s eta 0:00:00


In [2]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from wilds import get_dataset
from wilds.common.data_loaders import get_train_loader, get_eval_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

/usr/local/lib/python3.12/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


cuda


In [3]:
root_dir = '/kaggle/input/datasets/rayyanshuda/waterbirds-wilds-v1/data'
dataset = get_dataset(dataset='waterbirds', download=True, root_dir=root_dir)
print(f"Total examples: {len(dataset)}")  # should be 11788

Total examples: 11788


In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),   # tuple, not an int for exact square resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = dataset.get_subset('train', transform=transform)
val_data = dataset.get_subset('val', transform=transform)
test_data = dataset.get_subset('test', transform=transform)

print(len(train_data), len(val_data), len(test_data))  # should be 4795 1199 5794

4795 1199 5794


In [5]:
from wilds.common.data_loaders import get_train_loader, get_eval_loader

BATCH_SIZE = 128
grouper = dataset._eval_grouper

train_loader = get_train_loader(
    'standard', train_data, batch_size=BATCH_SIZE,
    uniform_over_groups=True, grouper=grouper,
)  # matches the reference --reweight_groups: WeightedRandomSampler by inverse group frequency
val_loader = get_eval_loader('standard', val_data, batch_size=BATCH_SIZE)
test_loader = get_eval_loader('standard', test_data, batch_size=BATCH_SIZE)

In [6]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 230MB/s]


In [7]:
n_groups = grouper.n_groups
group_names = ['landbird/land', 'landbird/water', 'waterbird/land', 'waterbird/water']

class GroupDROLoss:
    def __init__(self, n_groups, step_size=0.01, device='cpu'):
        self.adv_probs = torch.ones(n_groups, device=device) / n_groups
        self.step_size = step_size
        self.n_groups = n_groups

    def __call__(self, per_sample_losses, group_idx):
        group_count = torch.zeros(self.n_groups, device=per_sample_losses.device)
        group_count.scatter_add_(0, group_idx, torch.ones_like(group_idx, dtype=torch.float))
        group_loss = torch.zeros(self.n_groups, device=per_sample_losses.device)
        group_loss.scatter_add_(0, group_idx, per_sample_losses)
        group_loss = group_loss / group_count.clamp(min=1)

        self.adv_probs = self.adv_probs * torch.exp(self.step_size * group_loss.detach())
        self.adv_probs = self.adv_probs / self.adv_probs.sum()

        robust_loss = (group_loss * self.adv_probs).sum()
        return robust_loss, group_loss.detach()

criterion_none = nn.CrossEntropyLoss(reduction='none')
dro_loss = GroupDROLoss(n_groups=n_groups, step_size=0.01, device=device)

train_groups = grouper.metadata_to_group(train_data.metadata_array)
train_group_counts = torch.zeros(n_groups, dtype=torch.float)
train_group_counts.scatter_add_(0, train_groups, torch.ones_like(train_groups, dtype=torch.float))
print("Train group counts:", train_group_counts.tolist())  # should be [3498, 184, 56, 1057]

def evaluate(model, loader):
    model.eval()
    all_preds, all_y, all_metadata = [], [], []
    with torch.no_grad():
        for x, y, metadata in loader:
            x = x.to(device)
            logits = model(x)
            all_preds.append(logits.argmax(dim=1).cpu())
            all_y.append(y)
            all_metadata.append(metadata)
    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_y)
    metadata = torch.cat(all_metadata)

    g = grouper.metadata_to_group(metadata)
    correct = (y_pred == y_true).float()
    group_n = torch.zeros(n_groups, dtype=torch.float)
    group_n.scatter_add_(0, g, torch.ones_like(g, dtype=torch.float))
    group_correct = torch.zeros(n_groups, dtype=torch.float)
    group_correct.scatter_add_(0, g, correct)
    group_acc = group_correct / group_n.clamp(min=1)

    worst_group_acc = group_acc[group_n > 0].min().item()
    adj_acc_avg = (group_acc * train_group_counts).sum().item() / train_group_counts.sum().item()

    lines = [f"  {name}: {acc:.3f} (n={int(n)})"
             for name, acc, n in zip(group_names, group_acc.tolist(), group_n.tolist())]
    results_str = (f"Adjusted average acc: {adj_acc_avg:.3f}\n"
                    f"Worst-group acc: {worst_group_acc:.3f}\n" + "\n".join(lines))
    return adj_acc_avg, worst_group_acc, results_str

Train group counts: [3498.0, 184.0, 56.0, 1057.0]


In [8]:
sanity_subset = Subset(train_data, list(range(256)))
sanity_loader = DataLoader(sanity_subset, batch_size=32, shuffle=True)

sanity_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
sanity_model.fc = nn.Linear(sanity_model.fc.in_features, 2)
sanity_model = sanity_model.to(device)
sanity_optimizer = optim.SGD(sanity_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
sanity_dro = GroupDROLoss(n_groups=n_groups, step_size=0.01, device=device)

for epoch in range(10):
    sanity_model.train()
    total_loss = 0.0
    for x, y, metadata in sanity_loader:      # or train_loader in cell 9
        x, y = x.to(device), y.to(device)
        g = grouper.metadata_to_group(metadata).to(device)   # metadata itself stays on CPU here
        sanity_optimizer.zero_grad()                          # or optimizer.zero_grad() in cell 9
        per_sample = criterion_none(sanity_model(x), y)       # or model(x) in cell 9
        loss, _ = sanity_dro(per_sample, g)                   # or dro_loss(per_sample, g) in cell 9
        loss.backward()
        sanity_optimizer.step()                                # or optimizer.step() in cell 9
        total_loss += loss.item() * x.size(0)
    print(f"Sanity epoch {epoch+1}: loss = {total_loss / len(sanity_subset):.4f}")

sanity_avg, sanity_wg, sanity_str = evaluate(sanity_model, sanity_loader)
print("\nSanity eval check:\n" + sanity_str)  # just confirms .eval() runs cleanly, numbers dont mean muc

Sanity epoch 1: loss = 0.5821
Sanity epoch 2: loss = 0.3764
Sanity epoch 3: loss = 0.2459
Sanity epoch 4: loss = 0.1678
Sanity epoch 5: loss = 0.0937
Sanity epoch 6: loss = 0.0567
Sanity epoch 7: loss = 0.0415
Sanity epoch 8: loss = 0.0293
Sanity epoch 9: loss = 0.0218
Sanity epoch 10: loss = 0.0189

Sanity eval check:
Adjusted average acc: 1.000
Worst-group acc: 1.000
  landbird/land: 1.000 (n=85)
  landbird/water: 1.000 (n=4)
  waterbird/land: 1.000 (n=9)
  waterbird/water: 1.000 (n=158)


In [9]:
NUM_EPOCHS = 300
best_wg_acc, best_avg_acc = -1, -1
best_wg_state, best_avg_state = None, None

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y, metadata in train_loader:
        x, y = x.to(device), y.to(device)
        g = grouper.metadata_to_group(metadata).to(device)
        optimizer.zero_grad()
        per_sample = criterion_none(model(x), y)
        loss, _ = dro_loss(per_sample, g)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_data)
    val_avg_acc, val_wg_acc, _ = evaluate(model, val_loader)

    if val_wg_acc > best_wg_acc:
        best_wg_acc = val_wg_acc
        best_wg_state = copy.deepcopy(model.state_dict())
    if val_avg_acc > best_avg_acc:
        best_avg_acc = val_avg_acc
        best_avg_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | robust_loss={train_loss:.4f} | "
          f"val_avg_acc={val_avg_acc:.4f} | val_wg_acc={val_wg_acc:.4f}")

torch.save(best_wg_state, '/kaggle/working/dro_best_worst_group.pt')
torch.save(best_avg_state, '/kaggle/working/dro_best_avg.pt')

Epoch 1/300 | robust_loss=0.4580 | val_avg_acc=0.9298 | val_wg_acc=0.8120
Epoch 2/300 | robust_loss=0.1609 | val_avg_acc=0.9498 | val_wg_acc=0.8195
Epoch 3/300 | robust_loss=0.0805 | val_avg_acc=0.9648 | val_wg_acc=0.7820
Epoch 4/300 | robust_loss=0.0504 | val_avg_acc=0.9660 | val_wg_acc=0.7519
Epoch 5/300 | robust_loss=0.0296 | val_avg_acc=0.9654 | val_wg_acc=0.7669
Epoch 6/300 | robust_loss=0.0230 | val_avg_acc=0.9615 | val_wg_acc=0.7744
Epoch 7/300 | robust_loss=0.0159 | val_avg_acc=0.9615 | val_wg_acc=0.7895
Epoch 8/300 | robust_loss=0.0121 | val_avg_acc=0.9679 | val_wg_acc=0.7368
Epoch 9/300 | robust_loss=0.0088 | val_avg_acc=0.9675 | val_wg_acc=0.7293
Epoch 10/300 | robust_loss=0.0080 | val_avg_acc=0.9672 | val_wg_acc=0.7293
Epoch 11/300 | robust_loss=0.0061 | val_avg_acc=0.9693 | val_wg_acc=0.7293
Epoch 12/300 | robust_loss=0.0064 | val_avg_acc=0.9645 | val_wg_acc=0.7744
Epoch 13/300 | robust_loss=0.0048 | val_avg_acc=0.9677 | val_wg_acc=0.7218
Epoch 14/300 | robust_loss=0.0042 

In [10]:
model.load_state_dict(best_wg_state)
avg1, wg1, str1 = evaluate(model, test_loader)
print("Best val worst-group acc")
print(str1)
print(f"Test avg acc: {avg1:.4f}, Test worst-group acc: {wg1:.4f}")

model.load_state_dict(best_avg_state)
avg2, wg2, str2 = evaluate(model, test_loader)
print("\nBest val average acc")
print(str2)
print(f"Test avg acc: {avg2:.4f}, Test worst-group acc: {wg2:.4f}")

Best val worst-group acc
Adjusted average acc: 0.954
Worst-group acc: 0.861
  landbird/land: 0.967 (n=2255)
  landbird/water: 0.879 (n=2255)
  waterbird/land: 0.861 (n=642)
  waterbird/water: 0.928 (n=642)
Test avg acc: 0.9540, Test worst-group acc: 0.8614

Best val average acc
Adjusted average acc: 0.976
Worst-group acc: 0.707
  landbird/land: 0.998 (n=2255)
  landbird/water: 0.875 (n=2255)
  waterbird/land: 0.707 (n=642)
  waterbird/water: 0.935 (n=642)
Test avg acc: 0.9757, Test worst-group acc: 0.7072
